In [1]:
import os, glob, re
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 100) 

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Point this at the Kaggle dataset folder
# DATA_DIR = "/kaggle/input/march-machine-learning-mania-2025"
DATA_DIR = "data/"

def find_file(pattern):
    matches = glob.glob(os.path.join(DATA_DIR, "**", pattern), recursive=True)
    if not matches:
        raise FileNotFoundError(f"Could not find {pattern} under {DATA_DIR}")
    # pick shortest path (usually the canonical one)
    return sorted(matches, key=len)[0]

def find_file_optional(pattern):
    matches = glob.glob(os.path.join(DATA_DIR, "**", pattern), recursive=True)
    if not matches:
        return None
    return sorted(matches, key=len)[0]

w_reg = find_file("WRegularSeasonDetailedResults.csv")
w_tour = find_file("WNCAATourneyDetailedResults.csv")
w_seeds = find_file("WNCAATourneySeeds.csv")
w_teams = find_file("WTeams.csv")
w_team_conferences = find_file("WTeamConferences.csv")
massey_path = find_file_optional("WMasseyOrdinals.csv")
sub_path = find_file("SampleSubmissionStage2.csv")


In [3]:
w_teams_df = pd.read_csv(w_teams)
w_team_conferences_df = pd.read_csv(w_team_conferences)
w_teams_df.head()

,TeamID,TeamName
0,3101,Abilene Chr
1,3102,Air Force
2,3103,Akron
3,3104,Alabama
4,3105,Alabama A&M


In [4]:
purdue_team_id = 1345
w_teams_df.query(f"TeamID == {purdue_team_id}")

,TeamID,TeamName


# 1) Load + parse seeds into numeric features

We’ll extract:
- SeedNum (1–16)
- optional Region (W/X/Y/Z) as a small categorical

In [5]:
seeds_path = find_file("WNCAATourneySeeds.csv")
seeds = pd.read_csv(seeds_path)

# Seed string examples: "W01", "X16a", "Z11b"
def parse_seed(seed_str: str):
    # region = first letter
    region = seed_str[0]
    # numeric seed = next two digits
    seed_num = int(seed_str[1:3])
    # play-in suffix may exist (a/b), otherwise ""
    play = seed_str[3:] if len(seed_str) > 3 else ""
    return region, seed_num, play

tmp = seeds["Seed"].apply(parse_seed)
seeds["Region"] = tmp.apply(lambda x: x[0])
seeds["SeedNum"] = tmp.apply(lambda x: x[1]).astype(np.int16)

# Lookup dict: (Season, TeamID) -> SeedNum (and Region)
seednum_map = {(int(r.Season), int(r.TeamID)): int(r.SeedNum) for r in seeds.itertuples()}
region_map  = {(int(r.Season), int(r.TeamID)): str(r.Region)  for r in seeds.itertuples()}

seeds.head()


,Season,Seed,TeamID,Region,SeedNum
0,1998,W01,3330,W,1
1,1998,W02,3163,W,2
2,1998,W03,3112,W,3
3,1998,W04,3301,W,4
4,1998,W05,3272,W,5


# 2) Build per-team per-game rows from RegularSeasonDetailedResults

We’ll convert each game into two “team perspective” rows:

- one for the winner (label “team” stats = W*, “opp” stats = L*)

- one for the loser (label “team” stats = L*, “opp” stats = W*)

This makes sequences easy.

In [6]:
reg = pd.read_csv(w_reg)

# Common detailed stat columns in this dataset (winner/loser prefixed)
# We'll use only columns that exist to be robust.
W_cols = [c for c in reg.columns if c.startswith("W") & (c not in ['WTeamID', 'WLoc'])]
L_cols = [c for c in reg.columns if c.startswith("L") & (c not in ['LTeamID', 'WLoc'])]

id_cols = [c for c in ["Season", "DayNum", "WTeamID", "LTeamID", "WLoc", "NumOT"] if c in reg.columns]

reg = reg[id_cols + sorted(set(W_cols + L_cols))].copy()
reg.head()


,Season,DayNum,WTeamID,LTeamID,WLoc,NumOT,LAst,LBlk,LDR,LFGA,LFGA3,LFGM,LFGM3,LFTA,LFTM,LOR,LPF,LScore,LStl,LTO,WAst,WBlk,WDR,WFGA,WFGA3,WFGM,WFGM3,WFTA,WFTM,WOR,WPF,WScore,WStl,WTO
0,2010,11,3103,3237,H,0,11,6,27,54,13,20,3,10,6,11,19,49,7,23,14,0,26,54,9,23,5,19,12,10,15,63,7,18
1,2010,11,3104,3399,N,0,7,2,26,63,21,25,4,27,14,14,27,68,4,20,15,2,31,62,12,26,5,28,16,16,25,73,5,20
2,2010,11,3110,3224,A,0,8,0,23,58,14,19,2,23,19,17,15,59,6,15,18,2,23,62,15,29,6,12,7,14,17,71,6,13
3,2010,11,3111,3267,A,0,15,5,22,74,26,18,6,25,16,22,14,58,14,11,14,10,40,52,11,27,4,9,5,6,18,63,5,27
4,2010,11,3119,3447,H,1,12,2,32,74,17,25,9,21,11,21,14,70,4,14,18,3,33,74,20,30,7,11,7,14,18,74,5,11


# Feature builder

We’ll produce features like:

- team box score stats
- opponent stats
- margin
- location one-hot (H/A/N from the team’s POV)

In [7]:
import re
import numpy as np
import pandas as pd

def make_team_game_rows(reg_df: pd.DataFrame, diff_suffix: str = "dif") -> pd.DataFrame:
    df = reg_df.copy()

    # Winner perspective
    w = pd.DataFrame({
        "Season": df["Season"].values,
        "DayNum": df["DayNum"].values,
        "TeamID": df["WTeamID"].values,
        "OppID": df["LTeamID"].values,
        "IsWin": 1,
        "WLoc": df["WLoc"].values if "WLoc" in df.columns else "N",
    })

    # Loser perspective
    l = pd.DataFrame({
        "Season": df["Season"].values,
        "DayNum": df["DayNum"].values,
        "TeamID": df["LTeamID"].values,
        "OppID": df["WTeamID"].values,
        "IsWin": 0,
        "WLoc": df["WLoc"].values if "WLoc" in df.columns else "N",
    })

    # Attach stats: for winner row, team stats are W*, opp stats are L*
    # for loser row, team stats are L*, opp stats are W*
    stat_names = []
    EXCLUDE_STATS = {
        "TeamID", "OppID", "Season", "DayNum", "NumOT", "WLoc"
    }

    stat_names = []
    for c in df.columns:
        if re.match(r"^[WL][A-Z].*", c):
            base = c[1:]  # strip W/L prefix
            if base not in EXCLUDE_STATS:
                stat_names.append(base)
    
    stat_names = sorted(set(stat_names))


    for s in stat_names:
        w_team = "W" + s
        w_opp  = "L" + s
        l_team = "L" + s
        l_opp  = "W" + s

        if w_team in df.columns and w_opp in df.columns:
            w[f"T_{s}"] = df[w_team].values
            w[f"O_{s}"] = df[w_opp].values
        if l_team in df.columns and l_opp in df.columns:
            l[f"T_{s}"] = df[l_team].values
            l[f"O_{s}"] = df[l_opp].values

    out = pd.concat([w, l], axis=0, ignore_index=True)

    # Location from Team's POV:
    def loc_from_team_pov(is_win, wloc):
        if wloc not in ("H", "A", "N"):
            return "N"
        if wloc == "N":
            return "N"
        if is_win == 1:
            return wloc
        return "A" if wloc == "H" else "H"

    out["Loc"] = [loc_from_team_pov(w, wl) for w, wl in zip(out["IsWin"], out["WLoc"])]

    # Simple derived features: Margin (tries multiple name variants)
    if "T_Score" in out.columns and "O_Score" in out.columns:
        out["Margin"] = pd.to_numeric(out["T_Score"], errors="coerce") - pd.to_numeric(out["O_Score"], errors="coerce")
    else:
        score_candidates = [c for c in out.columns if c.lower() in ("t_score","t_pts","t_points")]
        opp_candidates = [c for c in out.columns if c.lower() in ("o_score","o_pts","o_points")]
        if score_candidates and opp_candidates:
            out["Margin"] = pd.to_numeric(out[score_candidates[0]], errors="coerce") - pd.to_numeric(out[opp_candidates[0]], errors="coerce")
        else:
            out["Margin"] = 0.0

    # One-hot location
    out["Loc_H"] = (out["Loc"] == "H").astype(np.int8)
    out["Loc_A"] = (out["Loc"] == "A").astype(np.int8)
    out["Loc_N"] = (out["Loc"] == "N").astype(np.int8)

    # --- NEW: Derived difference features for each stat ---
    # For each stat in stat_names, if T_<stat> and O_<stat> exist, create <stat>_<diff_suffix>
    for s in stat_names:
        tcol = f"T_{s}"
        ocol = f"O_{s}"
        dif_col = f"{s}_{diff_suffix}"
        if tcol in out.columns and ocol in out.columns:
            # ensure numeric, coerce non-numeric to NaN, then compute difference
            out[tcol] = pd.to_numeric(out[tcol], errors="coerce")
            out[ocol] = pd.to_numeric(out[ocol], errors="coerce")
            out[dif_col] = out[tcol] - out[ocol]

    # Keep only numeric features + keys (same as before)
    # If you prefer to keep Loc and WLoc, remove them from this filter
    numeric_and_keys = ["Season", "DayNum", "TeamID", "OppID", "IsWin", "WLoc", "Loc", "Loc_H", "Loc_A", "Loc_N"]
    # add any other non-numeric keys you want preserved
    remaining = [c for c in out.columns if (pd.api.types.is_numeric_dtype(out[c]) or c in numeric_and_keys)]
    out = out[remaining]

    return out.sort_values(["Season","TeamID","DayNum"]).reset_index(drop=True)

team_games = make_team_game_rows(reg)
team_games.head()


,Season,DayNum,TeamID,OppID,IsWin,WLoc,T_Ast,O_Ast,T_Blk,O_Blk,T_DR,O_DR,T_FGA,O_FGA,T_FGA3,O_FGA3,T_FGM,O_FGM,T_FGM3,O_FGM3,T_FTA,O_FTA,T_FTM,O_FTM,T_OR,O_OR,T_PF,O_PF,T_Score,O_Score,T_Stl,O_Stl,T_TO,O_TO,Loc,Margin,Loc_H,Loc_A,Loc_N,Ast_dif,Blk_dif,DR_dif,FGA_dif,FGA3_dif,FGM_dif,FGM3_dif,FTA_dif,FTM_dif,OR_dif,PF_dif,Score_dif,Stl_dif,TO_dif
0,2010,11,3102,3394,0,H,8,13,0,1,21,24,47,64,17,18,15,25,5,2,20,18,11,13,12,21,19,16,46,65,10,10,25,18,A,-19,0,1,0,-5,-1,-3,-17,-1,-10,3,2,-2,-9,3,-19,0,7
1,2010,12,3102,3399,0,N,5,19,0,3,26,27,51,64,9,31,14,29,1,13,29,19,20,10,17,15,16,26,49,81,1,4,15,7,N,-32,0,0,1,-14,-3,-1,-13,-22,-15,-12,10,10,2,-10,-32,-3,8
2,2010,18,3102,3339,0,A,13,21,0,6,23,23,59,63,22,32,26,26,8,12,7,11,5,9,11,16,14,8,65,73,6,4,12,10,H,-8,1,0,0,-8,-6,0,-4,-10,0,-4,-4,-4,-5,6,-8,2,2
3,2010,23,3102,3119,0,H,6,15,1,6,24,27,60,56,16,18,16,23,3,7,10,10,7,7,19,11,14,12,42,60,7,10,17,12,A,-18,0,1,0,-9,-5,-3,4,-2,-7,-4,0,0,8,2,-18,-3,5
4,2010,25,3102,3392,0,H,5,13,0,4,21,18,53,52,10,14,20,22,0,4,29,36,20,24,20,14,31,26,60,72,8,13,31,22,A,-12,0,1,0,-8,-4,3,1,-4,-2,-4,-7,-4,6,5,-12,-5,9


In [8]:
team_games.query(f"TeamID == {purdue_team_id}").tail()

,Season,DayNum,TeamID,OppID,IsWin,WLoc,T_Ast,O_Ast,T_Blk,O_Blk,T_DR,O_DR,T_FGA,O_FGA,T_FGA3,O_FGA3,T_FGM,O_FGM,T_FGM3,O_FGM3,T_FTA,O_FTA,T_FTM,O_FTM,T_OR,O_OR,T_PF,O_PF,T_Score,O_Score,T_Stl,O_Stl,T_TO,O_TO,Loc,Margin,Loc_H,Loc_A,Loc_N,Ast_dif,Blk_dif,DR_dif,FGA_dif,FGA3_dif,FGM_dif,FGM3_dif,FTA_dif,FTM_dif,OR_dif,PF_dif,Score_dif,Stl_dif,TO_dif


In [9]:
w_teams_df.query("TeamID == 1276")

,TeamID,TeamName


In [10]:
 # Net Efficiency

In [11]:
def add_possession_efficiency_features(df):
    out = df.copy()

    required = [
        "T_FGA", "T_OR", "T_TO", "T_FTA", "T_Score",
        "O_FGA", "O_OR", "O_TO", "O_FTA", "O_Score"
    ]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"Missing required columns for possessions: {missing}")

    # Possession estimates
    out["T_Poss"] = out["T_FGA"] - out["T_OR"] + out["T_TO"] + 0.475 * out["T_FTA"]
    out["O_Poss"] = out["O_FGA"] - out["O_OR"] + out["O_TO"] + 0.475 * out["O_FTA"]

    # Average possessions (shared denominator)
    out["Poss"] = 0.5 * (out["T_Poss"] + out["O_Poss"]).clip(lower=1.0)

    # Team efficiencies
    out["T_OffEff"] = 100.0 * out["T_Score"] / out["Poss"]
    out["T_DefEff"] = 100.0 * out["O_Score"] / out["Poss"]
    out["T_NetEff"] = out["T_OffEff"] - out["T_DefEff"]

    # Opponent efficiencies from THEIR POV (swap scores)
    out["O_OffEff"] = 100.0 * out["O_Score"] / out["Poss"]
    out["O_DefEff"] = 100.0 * out["T_Score"] / out["Poss"]
    out["O_NetEff"] = out["O_OffEff"] - out["O_DefEff"]

    # Differences (Team - Opp)
    out["OffEff_dif"] = out["T_OffEff"] - out["O_OffEff"]
    out["DefEff_dif"] = out["T_DefEff"] - out["O_DefEff"]
    out["NetEff_dif"] = out["T_NetEff"] - out["O_NetEff"]

    return out

def add_rolling_neteff_features_team_and_opp(df, windows=(5, 10), feature_col="T_NetEff"):
    """
    Adds rolling mean/std of a per-game efficiency column for:
      - the row TeamID (prefix T_)
      - the row OppID (prefix O_)
    Then adds diffs: T - O for each rolling feature.
    Uses only prior games (shift(1)).
    """
    out = df.copy()
    out = out.sort_values(["Season", "TeamID", "DayNum"], kind="mergesort")

    if feature_col not in out.columns:
        raise ValueError(f"Missing feature_col='{feature_col}'. Did you run add_possession_efficiency_features first?")

    # ---- Team rolling
    grpT = out.groupby(["Season", "TeamID"], sort=False)

    for w in windows:
        out[f"T_NetEff_Mean_{w}"] = grpT[feature_col].transform(
            lambda s: s.shift(1).rolling(w, min_periods=1).mean()
        )
        out[f"T_NetEff_Std_{w}"] = grpT[feature_col].transform(
            lambda s: s.shift(1).rolling(w, min_periods=1).std()
        )

    for c in out.columns:
        if c.startswith("T_NetEff_Std_"):
            out[c] = out[c].fillna(0.0)

    # ---- Opp rolling: compute same rollups but keyed by (Season, OppID)
    # Build a thin frame so we can merge back cleanly by row identity.
    base_keys = ["Season", "DayNum", "TeamID", "OppID"]
    tmp = out[base_keys + [feature_col]].copy()

    # Treat OppID as "TeamID" for grouping
    tmp = tmp.rename(columns={"OppID": "TeamID", "TeamID": "OrigTeamID"})
    tmp = tmp.sort_values(["Season", "TeamID", "DayNum"], kind="mergesort")
    grpO = tmp.groupby(["Season", "TeamID"], sort=False)

    opp_roll = tmp[["Season", "DayNum", "OrigTeamID", "TeamID"]].copy()  # OrigTeamID is original TeamID; TeamID is original OppID
    opp_roll = opp_roll.rename(columns={"OrigTeamID": "TeamID", "TeamID": "OppID"})  # back to original key names

    for w in windows:
        opp_roll[f"O_NetEff_Mean_{w}"] = grpO[feature_col].transform(
            lambda s: s.shift(1).rolling(w, min_periods=1).mean()
        ).values
        opp_roll[f"O_NetEff_Std_{w}"] = grpO[feature_col].transform(
            lambda s: s.shift(1).rolling(w, min_periods=1).std()
        ).values

    for c in opp_roll.columns:
        if c.startswith("O_NetEff_Std_"):
            opp_roll[c] = opp_roll[c].fillna(0.0)

    # Merge opponent rolling features back
    out = out.merge(opp_roll, on=["Season", "DayNum", "TeamID", "OppID"], how="left")

    # ---- Diffs (Team - Opp) for rolling features
    for w in windows:
        out[f"NetEff_Mean_dif_{w}"] = out[f"T_NetEff_Mean_{w}"] - out[f"O_NetEff_Mean_{w}"]
        out[f"NetEff_Std_dif_{w}"]  = out[f"T_NetEff_Std_{w}"]  - out[f"O_NetEff_Std_{w}"]

    return out

team_games = add_possession_efficiency_features(team_games)
team_games = add_rolling_neteff_features_team_and_opp(team_games, windows=(5, 10), feature_col="T_NetEff")
team_games.head()

,Season,DayNum,TeamID,OppID,IsWin,WLoc,T_Ast,O_Ast,T_Blk,O_Blk,T_DR,O_DR,T_FGA,O_FGA,T_FGA3,O_FGA3,T_FGM,O_FGM,T_FGM3,O_FGM3,T_FTA,O_FTA,T_FTM,O_FTM,T_OR,O_OR,T_PF,O_PF,T_Score,O_Score,T_Stl,O_Stl,T_TO,O_TO,Loc,Margin,Loc_H,Loc_A,Loc_N,Ast_dif,Blk_dif,DR_dif,FGA_dif,FGA3_dif,FGM_dif,FGM3_dif,FTA_dif,FTM_dif,OR_dif,PF_dif,Score_dif,Stl_dif,TO_dif,T_Poss,O_Poss,Poss,T_OffEff,T_DefEff,T_NetEff,O_OffEff,O_DefEff,O_NetEff,OffEff_dif,DefEff_dif,NetEff_dif,T_NetEff_Mean_5,T_NetEff_Std_5,T_NetEff_Mean_10,T_NetEff_Std_10,O_NetEff_Mean_5,O_NetEff_Std_5,O_NetEff_Mean_10,O_NetEff_Std_10,NetEff_Mean_dif_5,NetEff_Std_dif_5,NetEff_Mean_dif_10,NetEff_Std_dif_10
0,2010,11,3102,3394,0,H,8,13,0,1,21,24,47,64,17,18,15,25,5,2,20,18,11,13,12,21,19,16,46,65,10,10,25,18,A,-19,0,1,0,-5,-1,-3,-17,-1,-10,3,2,-2,-9,3,-19,0,7,69.500,69.550,69.5250,66.163251,93.491550,-27.328299,93.491550,66.163251,27.328299,-27.328299,27.328299,-54.656598,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.000000
1,2010,12,3102,3399,0,N,5,19,0,3,26,27,51,64,9,31,14,29,1,13,29,19,20,10,17,15,16,26,49,81,1,4,15,7,N,-32,0,0,1,-14,-3,-1,-13,-22,-15,-12,10,10,2,-10,-32,-3,8,62.775,65.025,63.9000,76.682316,126.760563,-50.078247,126.760563,76.682316,50.078247,-50.078247,50.078247,-100.156495,-27.328299,0.000000,-27.328299,0.000000,6.206362,0.000000,6.206362,0.000000,-33.534661,0.000000,-33.534661,0.000000
2,2010,18,3102,3339,0,A,13,21,0,6,23,23,59,63,22,32,26,26,8,12,7,11,5,9,11,16,14,8,65,73,6,4,12,10,H,-8,1,0,0,-8,-6,0,-4,-10,0,-4,-4,-4,-5,6,-8,2,2,63.325,62.225,62.7750,103.544405,116.288331,-12.743927,116.288331,103.544405,12.743927,-12.743927,12.743927,-25.487853,-38.703273,16.086643,-38.703273,16.086643,9.557282,1.282797,9.557282,1.282797,-48.260555,14.803846,-48.260555,14.803846
3,2010,23,3102,3119,0,H,6,15,1,6,24,27,60,56,16,18,16,23,3,7,10,10,7,7,19,11,14,12,42,60,7,10,17,12,A,-18,0,1,0,-9,-5,-3,4,-2,-7,-4,0,0,8,2,-18,-3,5,62.750,61.750,62.2500,67.469880,96.385542,-28.915663,96.385542,67.469880,28.915663,-28.915663,28.915663,-57.831325,-30.050158,18.815399,-30.050158,18.815399,6.488363,17.130996,6.488363,17.130996,-36.538520,1.684403,-36.538520,1.684403
4,2010,25,3102,3392,0,H,5,13,0,4,21,18,53,52,10,14,20,22,0,4,29,36,20,24,20,14,31,26,60,72,8,13,31,22,A,-12,0,1,0,-8,-4,3,1,-4,-2,-4,-7,-4,6,5,-12,-5,9,77.775,77.100,77.4375,77.481840,92.978208,-15.496368,92.978208,77.481840,15.496368,-15.496368,15.496368,-30.992736,-29.766534,15.373178,-29.766534,15.373178,30.659711,10.975891,30.659711,10.975891,-60.426245,4.397288,-60.426245,4.397288


In [12]:
# # Conference
# team_conf = w_team_conferences_df[["TeamID","ConfAbbrev"]].copy()
# team_conf_map = {
#     int(r.TeamID): r.ConfAbbrev
#     for r in team_conf.itertuples()
# }

# TOURNEY_CUTOFF_DAY = 133

# def compute_team_neteff_pre_tourney(team_games_df, cutoff_day=133):
#     df = team_games_df[team_games_df["DayNum"] <= cutoff_day]
#     return (
#         df.groupby(["Season","TeamID"])["T_NetEff"]
#         .mean()
#         .reset_index(name="Team_NetEff_Mean")
#     )

# team_neteff = compute_team_neteff_pre_tourney(team_games, TOURNEY_CUTOFF_DAY)

# team_neteff["Conf"] = team_neteff["TeamID"].map(team_conf_map)

# conf_neteff = (
#     team_neteff
#     .groupby(["Season","Conf"])["Team_NetEff_Mean"]
#     .mean()
#     .reset_index(name="Conf_NetEff")
# )

# def compute_conf_nonconf_winpct(reg_df, team_conf_map, cutoff_day=133):
#     df = reg_df[reg_df["DayNum"] <= cutoff_day].copy()

#     df["WConf"] = df["WTeamID"].map(team_conf_map)
#     df["LConf"] = df["LTeamID"].map(team_conf_map)

#     # Only games where conferences differ
#     nonconf = df[df["WConf"] != df["LConf"]]

#     w = nonconf[["Season","WConf"]].copy()
#     w["Conf"] = w["WConf"]
#     w["Win"] = 1

#     l = nonconf[["Season","LConf"]].copy()
#     l["Conf"] = l["LConf"]
#     l["Win"] = 0

#     games = pd.concat([
#         w[["Season","Conf","Win"]],
#         l[["Season","Conf","Win"]]
#     ])

#     agg = (
#         games.groupby(["Season","Conf"])
#         .agg(Wins=("Win","sum"), Games=("Win","count"))
#         .reset_index()
#     )

#     agg["Conf_NonConf_WinPct"] = agg["Wins"] / agg["Games"]
#     return agg[["Season","Conf","Conf_NonConf_WinPct"]]

# conf_nonconf = compute_conf_nonconf_winpct(reg, team_conf_map, TOURNEY_CUTOFF_DAY)

# conf_strength = (
#     conf_neteff
#     .merge(conf_nonconf, on=["Season","Conf"], how="left")
# )

# # Fill rare missing values
# conf_strength["Conf_NonConf_WinPct"] = conf_strength["Conf_NonConf_WinPct"].fillna(0.5)

# conf_neteff_map = {
#     (int(r.Season), r.Conf): float(r.Conf_NetEff)
#     for r in conf_strength.itertuples()
# }

# conf_winpct_map = {
#     (int(r.Season), r.Conf): float(r.Conf_NonConf_WinPct)
#     for r in conf_strength.itertuples()
# }

# DEFAULT_CONF_NETEFF = 0.0
# DEFAULT_CONF_WINPCT = 0.5

# conf_strength.head()

# # conf_strength['Conf_NetEff'] = (conf_strength["Conf_NetEff"] - conf_strength["Conf_NetEff"].mean()) / conf_strength["Conf_NetEff"].std()
# conf_strength['Conf_NetEff'] = conf_strength["Conf_NetEff"] / conf_strength["Conf_NetEff"].abs().max()
# conf_strength.query("Season == 2025").sort_values('Conf_NetEff', ascending=False).head()


# 3) Tournament labels (training pairs)

We’ll train on historical tournament games from `MNCAATourneyDetailedResults.csv`, similarly converting to (TeamA, TeamB, label).

In [13]:
tour = pd.read_csv(w_tour)

needed = ["Season","DayNum","WTeamID","LTeamID"]
for c in needed:
    if c not in tour.columns:
        raise ValueError(f"Missing {c} in tourney data")

# Create directional samples: A vs B
# We'll include both directions for stability:
# (W vs L, y=1) and (L vs W, y=0)
t1 = tour[["Season","DayNum","WTeamID","LTeamID"]].copy()
t1.rename(columns={"WTeamID":"TeamA","LTeamID":"TeamB"}, inplace=True)
t1["y"] = 1

t2 = tour[["Season","DayNum","WTeamID","LTeamID"]].copy()
t2.rename(columns={"WTeamID":"TeamB","LTeamID":"TeamA"}, inplace=True)
t2["y"] = 0

train_pairs = pd.concat([t1,t2], ignore_index=True)
train_pairs = train_pairs.sort_values(["Season","DayNum"]).reset_index(drop=True)
train_pairs.head()

,Season,DayNum,TeamA,TeamB,y
0,2010,138,3124,3201,1
1,2010,138,3173,3395,1
2,2010,138,3181,3214,1
3,2010,138,3199,3256,1
4,2010,138,3207,3265,1


# Rankings

In [14]:
if massey_path is None:
    print('WMasseyOrdinals.csv not found; skipping Massey features.')
    HAS_MASSEY = False
    team_games_aug = team_games.copy()
else:
    HAS_MASSEY = True
    massey = pd.read_csv(massey_path)
    SYSTEMS = ["POM"]  # example; replace with ones you see in your file
    SYSTEMS = [s for s in SYSTEMS if s in set(massey["SystemName"].unique())]
    print('Using systems:', SYSTEMS)

    # Convert OrdinalRank to numeric
    massey = massey[massey["SystemName"].isin(SYSTEMS)].copy()
    massey["Rank"] = massey["OrdinalRank"].astype(int)
    massey["RatingNegRank"] = -massey["Rank"]   # higher = better

    team_games_aug = team_games.copy()

    def assert_sorted(df, cols, name):
        s = df[cols].values
        # not a perfect lexicographic check, but catches most issues fast
        if not df.sort_values(cols).reset_index(drop=True).equals(df.reset_index(drop=True)):
            raise ValueError(f"{name} is not sorted by {cols}")

    assert_sorted(team_games_aug, ["Season","TeamID","DayNum"], "team_games_aug")
    if SYSTEMS:
        tmp = massey[massey["SystemName"] == SYSTEMS[0]][["Season","TeamID","RankingDayNum"]].copy()
        tmp = tmp.sort_values(["Season","TeamID","RankingDayNum"]).reset_index(drop=True)
        assert_sorted(tmp, ["Season","TeamID","RankingDayNum"], "massey (one system)")
        print('Sorting looks OK.')

    def add_massey_asof_groupwise(team_games_df, massey_df, system_name,
                                  id_col="TeamID", prefix="T"):
        # Prepare left
        left = team_games_df.copy()
        left["Season"] = left["Season"].astype(np.int64)
        left[id_col] = left[id_col].astype(np.int64)
        left["DayNum"] = left["DayNum"].astype(np.int64)

        # Prepare right (single system)
        right = massey_df[massey_df["SystemName"] == system_name][
            ["Season","TeamID","RankingDayNum","OrdinalRank"]
        ].copy()
        right["Season"] = right["Season"].astype(np.int64)
        right["TeamID"] = right["TeamID"].astype(np.int64)
        right["RankingDayNum"] = right["RankingDayNum"].astype(np.int64)
        right["Rank"] = right["OrdinalRank"].astype(np.float32)
        right["RatingNegRank"] = -right["Rank"]

        # Drop exact duplicate day ranks within a group
        right = (right.sort_values(["Season","TeamID","RankingDayNum"], kind="mergesort")
                      .drop_duplicates(["Season","TeamID","RankingDayNum"], keep="last"))

        # Groupwise asof
        out_parts = []
        right_groups = {k: g for k, g in right.groupby(["Season","TeamID"], sort=False)}

        for k, gL in left.groupby(["Season", id_col], sort=False):
            gL = gL.sort_values("DayNum", kind="mergesort")

            # right is keyed by ("Season","TeamID"), so map id_col -> TeamID key
            gR = right_groups.get((k[0], k[1]), None)

            if gR is None or len(gR) == 0:
                gL[f"{system_name}_{prefix}_Rating"] = -255.0
                out_parts.append(gL)
                continue

            gR = gR.sort_values("RankingDayNum", kind="mergesort")

            merged = pd.merge_asof(
                gL,
                gR[["RankingDayNum","Rank","RatingNegRank"]],
                left_on="DayNum",
                right_on="RankingDayNum",
                direction="backward",
                allow_exact_matches=True
            )

            gL[f"{system_name}_{prefix}_Rating"] = merged["RatingNegRank"].values # .fillna(-255.0).values
            out_parts.append(gL)

        return pd.concat(out_parts, ignore_index=True)


    def add_team_and_opp_massey(team_games_df, massey_df, system_name):
        # Team rating
        out = add_massey_asof_groupwise(team_games_df, massey_df, system_name,
                                        id_col="TeamID", prefix="T")

        # Opp rating: run the same logic but treat OppID as the id column
        out = add_massey_asof_groupwise(out, massey_df, system_name,
                                        id_col="OppID", prefix="O")

        # Difference (Team - Opp). Since ratings are "negative rank",
        # higher is better and Team - Opp still works as "relative strength".
        out[f"{system_name}_Rating_Diff"] = out[f"{system_name}_T_Rating"] - out[f"{system_name}_O_Rating"]

        return out


    if SYSTEMS:
        team_games_aug = add_team_and_opp_massey(team_games_aug, massey, SYSTEMS[0])
        team_games_aug[[ "Season","DayNum","TeamID","OppID", f"{SYSTEMS[0]}_T_Rating", f"{SYSTEMS[0]}_O_Rating", f"{SYSTEMS[0]}_Rating_Diff" ]].head()



WMasseyOrdinals.csv not found; skipping Massey features.


In [15]:
# for sys in SYSTEMS:
#     team_games_aug[f'{sys}_T_Rating'] = team_games_aug.groupby(["Season", "TeamID"])[f'{sys}_T_Rating'].bfill()
#     team_games_aug[f'{sys}_O_Rating'] = team_games_aug.groupby(["Season", "TeamID"])[f'{sys}_O_Rating'].bfill()
#     team_games_aug[f'{sys}_Rating_Diff'] = team_games_aug[f'{sys}_T_Rating'] - team_games_aug[f'{sys}_O_Rating']

In [16]:
# team_games_aug.query("Season == 2025 & TeamID == 1345").sort_values('DayNum').tail()

In [17]:
if 'POM_T_Rating' in team_games_aug.columns:
    team_games_aug.query('Season == 2025 & TeamID == 1345').sort_values('DayNum').tail()[
        ['TeamID', 'OppID', 'T_NetEff', 'T_NetEff_Mean_5', 'O_NetEff', 'NetEff_dif', 'POM_T_Rating', 'POM_O_Rating', 'POM_Rating_Diff']
    ]


# Team Overall Win-Loss

In [18]:
TOURNEY_CUTOFF_DAY = 133

def build_pre_tourney_win_loss(reg_df, cutoff_day=133):
    """
    Returns a dataframe with one row per (Season, TeamID)
    containing wins, losses, win pct, and last-10 win pct
    prior to the tournament.
    """
    df = reg_df.copy()
    df = df[df["DayNum"] <= cutoff_day]

    # Winner perspective
    w = df[["Season","DayNum","WTeamID"]].copy()
    w["TeamID"] = w["WTeamID"]
    w["Win"] = 1

    # Loser perspective
    l = df[["Season","DayNum","LTeamID"]].copy()
    l["TeamID"] = l["LTeamID"]
    l["Win"] = 0

    games = pd.concat([w[["Season","DayNum","TeamID","Win"]],
                       l[["Season","DayNum","TeamID","Win"]]],
                       ignore_index=True)

    # Aggregate totals
    agg = (
        games
        .groupby(["Season","TeamID"])
        .agg(
            Wins=("Win","sum"),
            Games=("Win","count")
        )
        .reset_index()
    )

    agg["Losses"] = agg["Games"] - agg["Wins"]
    agg["WinPct"] = agg["Wins"] / agg["Games"]

    # --- Last 10 games win pct ---
    games = games.sort_values(["Season","TeamID","DayNum"])
    last10 = (
        games
        .groupby(["Season","TeamID"])["Win"]
        .apply(lambda s: s.tail(10).mean())
        .reset_index(name="Last10_WinPct")
    )

    out = agg.merge(last10, on=["Season","TeamID"], how="left")
    return out

wl_context = build_pre_tourney_win_loss(reg, cutoff_day=133)
wl_context.head()



,Season,TeamID,Wins,Games,Losses,WinPct,Last10_WinPct
0,2010,3102,1,28,27,0.035714,0.0
1,2010,3103,17,30,13,0.566667,0.6
2,2010,3104,11,29,18,0.379310,0.3
3,2010,3105,14,27,13,0.518519,0.6
4,2010,3106,12,29,17,0.413793,0.5


In [19]:
winpct_map = {
    (int(r.Season), int(r.TeamID)): float(r.WinPct)
    for r in wl_context.itertuples()
}

last10_map = {
    (int(r.Season), int(r.TeamID)): float(r.Last10_WinPct)
    for r in wl_context.itertuples()
}

DEFAULT_WINPCT = 0.5
DEFAULT_LAST10 = 0.5

# 3) Build sequences: (Season, TeamID) -> tensor [G, F]

We’ll pick a feature list from `team_games` that’s numeric and stable, then normalize with a scaler fit on training seasons only.

In [20]:
# Feature columns (numeric only, exclude IDs/keys)
key_cols = {"Season","DayNum","TeamID","OppID","WLoc","Loc"}
full_feature_cols = [c for c in team_games_aug.columns if c not in key_cols and pd.api.types.is_numeric_dtype(team_games_aug[c])]
# feature_cols = [c for c in team_games_aug.columns if (('dif' in c) | (c in ['IsWin', 'NetEff', 'POM_Rating_Diff', 'Loc_H', 'Loc_A',]))]
feature_cols = [
    'IsWin', 
    'Score_dif',
    # 'TO_dif',
    # 'Ast_dif',
    'POM_Rating_Diff', 
    'Loc_H', 
    # 'NetEff_dif', 
    'NetEff_Mean_dif_5', 
    'NetEff_Mean_dif_10',
]

if 'POM_Rating_Diff' not in team_games_aug.columns and 'POM_Rating_Diff' in feature_cols:
    feature_cols = [c for c in feature_cols if c != 'POM_Rating_Diff']


print("Num full features:", len(full_feature_cols))
print("Full features:", full_feature_cols)
print("\n")
print("Num features:", len(feature_cols))
print("Example features:", feature_cols)


Num full features: 71
Full features: ['IsWin', 'T_Ast', 'O_Ast', 'T_Blk', 'O_Blk', 'T_DR', 'O_DR', 'T_FGA', 'O_FGA', 'T_FGA3', 'O_FGA3', 'T_FGM', 'O_FGM', 'T_FGM3', 'O_FGM3', 'T_FTA', 'O_FTA', 'T_FTM', 'O_FTM', 'T_OR', 'O_OR', 'T_PF', 'O_PF', 'T_Score', 'O_Score', 'T_Stl', 'O_Stl', 'T_TO', 'O_TO', 'Margin', 'Loc_H', 'Loc_A', 'Loc_N', 'Ast_dif', 'Blk_dif', 'DR_dif', 'FGA_dif', 'FGA3_dif', 'FGM_dif', 'FGM3_dif', 'FTA_dif', 'FTM_dif', 'OR_dif', 'PF_dif', 'Score_dif', 'Stl_dif', 'TO_dif', 'T_Poss', 'O_Poss', 'Poss', 'T_OffEff', 'T_DefEff', 'T_NetEff', 'O_OffEff', 'O_DefEff', 'O_NetEff', 'OffEff_dif', 'DefEff_dif', 'NetEff_dif', 'T_NetEff_Mean_5', 'T_NetEff_Std_5', 'T_NetEff_Mean_10', 'T_NetEff_Std_10', 'O_NetEff_Mean_5', 'O_NetEff_Std_5', 'O_NetEff_Mean_10', 'O_NetEff_Std_10', 'NetEff_Mean_dif_5', 'NetEff_Std_dif_5', 'NetEff_Mean_dif_10', 'NetEff_Std_dif_10']


Num features: 5
Example features: ['IsWin', 'Score_dif', 'Loc_H', 'NetEff_Mean_dif_5', 'NetEff_Mean_dif_10']


Choose validation season split

Start with: last season as validation (you can change to rolling CV later).

In [21]:
train_start_season = 2005
val_season = 2024

all_seasons = sorted(train_pairs['Season'].unique())
if val_season not in all_seasons:
    raise ValueError(f'val_season={val_season} not in data. Available: {all_seasons[0]}..{all_seasons[-1]}')

train_seasons = [s for s in all_seasons if (s < val_season) and (s >= train_start_season)]

print(f'Train seasons: {min(train_seasons)} to {max(train_seasons)}')
print('Val season:', val_season)


Train seasons: 2010 to 2023
Val season: 2024


### Fit scaler on training seasons’ regular season rows

In [34]:
scaler = StandardScaler()

team_games_aug[feature_cols] = team_games_aug[feature_cols].fillna(0)
fit_df = team_games_aug[team_games_aug["Season"].isin(train_seasons)]
scaler.fit(fit_df[feature_cols].values)

team_games_scaled = team_games_aug.copy()
team_games_scaled[feature_cols] = scaler.transform(team_games_scaled[feature_cols].values)


In [35]:
 print("NaNs in team_games_aug:", team_games_aug[feature_cols].isna().sum().sort_values(ascending=False).head(10))

NaNs in team_games_aug: IsWin                 0
Score_dif             0
Loc_H                 0
NetEff_Mean_dif_5     0
NetEff_Mean_dif_10    0
dtype: int64


In [36]:
team_games_aug

,Season,DayNum,TeamID,OppID,IsWin,WLoc,T_Ast,O_Ast,T_Blk,O_Blk,T_DR,O_DR,T_FGA,O_FGA,T_FGA3,O_FGA3,T_FGM,O_FGM,T_FGM3,O_FGM3,T_FTA,O_FTA,T_FTM,O_FTM,T_OR,O_OR,T_PF,O_PF,T_Score,O_Score,T_Stl,O_Stl,T_TO,O_TO,Loc,Margin,Loc_H,Loc_A,Loc_N,Ast_dif,Blk_dif,DR_dif,FGA_dif,FGA3_dif,FGM_dif,FGM3_dif,FTA_dif,FTM_dif,OR_dif,PF_dif,Score_dif,Stl_dif,TO_dif,T_Poss,O_Poss,Poss,T_OffEff,T_DefEff,T_NetEff,O_OffEff,O_DefEff,O_NetEff,OffEff_dif,DefEff_dif,NetEff_dif,T_NetEff_Mean_5,T_NetEff_Std_5,T_NetEff_Mean_10,T_NetEff_Std_10,O_NetEff_Mean_5,O_NetEff_Std_5,O_NetEff_Mean_10,O_NetEff_Std_10,NetEff_Mean_dif_5,NetEff_Std_dif_5,NetEff_Mean_dif_10,NetEff_Std_dif_10
0,2010,11,3102,3394,0,H,8,13,0,1,21,24,47,64,17,18,15,25,5,2,20,18,11,13,12,21,19,16,46,65,10,10,25,18,A,-19,0,1,0,-5,-1,-3,-17,-1,-10,3,2,-2,-9,3,-19,0,7,69.500,69.550,69.5250,66.163251,93.491550,-27.328299,93.491550,66.163251,27.328299,-27.328299,27.328299,-54.656598,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000
1,2010,12,3102,3399,0,N,5,19,0,3,26,27,51,64,9,31,14,29,1,13,29,19,20,10,17,15,16,26,49,81,1,4,15,7,N,-32,0,0,1,-14,-3,-1,-13,-22,-15,-12,10,10,2,-10,-32,-3,8,62.775,65.025,63.9000,76.682316,126.760563,-50.078247,126.760563,76.682316,50.078247,-50.078247,50.078247,-100.156495,-27.328299,0.000000,-27.328299,0.000000,6.206362,0.000000,6.206362,0.000000,-33.534661,0.000000,-33.534661,0.000000
2,2010,18,3102,3339,0,A,13,21,0,6,23,23,59,63,22,32,26,26,8,12,7,11,5,9,11,16,14,8,65,73,6,4,12,10,H,-8,1,0,0,-8,-6,0,-4,-10,0,-4,-4,-4,-5,6,-8,2,2,63.325,62.225,62.7750,103.544405,116.288331,-12.743927,116.288331,103.544405,12.743927,-12.743927,12.743927,-25.487853,-38.703273,16.086643,-38.703273,16.086643,9.557282,1.282797,9.557282,1.282797,-48.260555,14.803846,-48.260555,14.803846
3,2010,23,3102,3119,0,H,6,15,1,6,24,27,60,56,16,18,16,23,3,7,10,10,7,7,19,11,14,12,42,60,7,10,17,12,A,-18,0,1,0,-9,-5,-3,4,-2,-7,-4,0,0,8,2,-18,-3,5,62.750,61.750,62.2500,67.469880,96.385542,-28.915663,96.385542,67.469880,28.915663,-28.915663,28.915663,-57.831325,-30.050158,18.815399,-30.050158,18.815399,6.488363,17.130996,6.488363,17.130996,-36.538520,1.684403,-36.538520,1.684403
4,2010,25,3102,3392,0,H,5,13,0,4,21,18,53,52,10,14,20,22,0,4,29,36,20,24,20,14,31,26,60,72,8,13,31,22,A,-12,0,1,0,-8,-4,3,1,-4,-2,-4,-7,-4,6,5,-12,-5,9,77.775,77.100,77.4375,77.481840,92.978208,-15.496368,92.978208,77.481840,15.496368,-15.496368,15.496368,-30.992736,-29.766534,15.373178,-29.766534,15.373178,30.659711,10.975891,30.659711,10.975891,-60.426245,4.397288,-60.426245,4.397288
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163411,2025,108,3480,3391,0,A,8,15,5,4,19,30,60,66,21,13,23,27,3,3,12,16,9,11,6,19,16,14,58,68,8,4,13,17,H,-10,1,0,0,-7,1,-11,-6,8,-4,0,-4,-2,-13,2,-10,4,-4,72.700,71.600,72.1500,80.388080,94.248094,-13.860014,94.248094,80.388080,13.860014,-13.860014,13.860014,-27.720028,1.503168,17.069802,-6.345504,20.922414,5.118246,16.208399,6.111274,19.861825,-3.615078,0.861403,-12.456778,1.060590
163412,2025,110,3480,3146,0,H,14,15,3,3,19,20,64,48,27,22,20,25,9,11,18,14,13,11,18,4,15,14,62,72,9,3,10,13,A,-10,0,1,0,-1,0,-1,16,5,-5,-2,4,2,14,1,-10,6,-3,64.550,63.650,64.1000,96.723869,112.324493,-15.600624,112.324493,96.723869,15.600624,-15.600624,15.600624,-31.201248,1.134771,17.450205,-6.785319,21.041094,-9.377096,25.477241,-11.356210,19.617323,10.511867,-8.027036,4.570891,1.423771
163413,2025,115,3480,3122,1,A,13,7,2,1,30,24,53,53,24,17,22,17,5,4,18,20,9,14,11,7,19,18,58,52,9,12,16,13,A,6,0,1,0,6,1,6,0,7,5,1,-2,-5,4,1,6,-3,3,66.550,68.500,67.5250,85.894113,77.008515,8.885598,77.008515,85.894113,-8.885598,8.885598,-8.885598,17.771196,2.233008,15.796048,-3.166880,14.553940,6.427575,12.643998,6.629571,13.474079,-4.194567

### Create dict of sequences

In [37]:
def build_team_seq_dict(df_scaled: pd.DataFrame, feature_cols, min_games=1):
    seq = {}
    grouped = df_scaled.groupby(["Season","TeamID"], sort=False)
    for (season, team), g in grouped:
        g = g.sort_values("DayNum")
        x = torch.tensor(g[feature_cols].values, dtype=torch.float32)
        if x.size(0) >= min_games:
            seq[(int(season), int(team))] = x
    return seq

team_seq = build_team_seq_dict(team_games_scaled, feature_cols, min_games=1)
print("Sequences built:", len(team_seq))

Sequences built: 5602


# 4) Dataset + padding

We’ll use last T games (e.g., 20). Teams with fewer than T games get zero-padding at the start.

In [38]:
class MatchupSeqDataset(Dataset):
    def __init__(
        self,
        pairs_df,
        team_seq_dict,
        seednum_map,
        winpct_map,
        last10_map,
        # team_conf_map,
        # conf_neteff_map,
        # conf_winpct_map,
        T,
        feature_dim
    ):
        self.df = pairs_df.reset_index(drop=True)
        self.team_seq = team_seq_dict
        self.seednum_map = seednum_map
        self.winpct_map = winpct_map
        self.last10_map = last10_map
        # self.team_conf_map = team_conf_map
        # self.conf_neteff_map = conf_neteff_map
        # self.conf_winpct_map = conf_winpct_map
        self.T = T
        self.F = feature_dim

        keep = []
        for _, r in self.df.iterrows():
            kA = (int(r.Season), int(r.TeamA))
            kB = (int(r.Season), int(r.TeamB))
            keep.append(
                kA in team_seq_dict and
                kB in team_seq_dict and
                kA in seednum_map and
                kB in seednum_map
            )
        self.df = self.df[np.array(keep)].reset_index(drop=True)

    def _pad_trunc(self, x):
        if x.size(0) >= self.T:
            return x[-self.T:]
        return torch.cat([torch.zeros(self.T - x.size(0), self.F), x], dim=0)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        season = int(r.Season)

        A = int(r.TeamA)
        B = int(r.TeamB)

        kA = (season, A)
        kB = (season, B)

        seqA = self._pad_trunc(self.team_seq[kA]).float()
        seqB = self._pad_trunc(self.team_seq[kB]).float()

        seedA = torch.tensor(self.seednum_map[kA], dtype=torch.float32)
        seedB = torch.tensor(self.seednum_map[kB], dtype=torch.float32)

        winA = torch.tensor(self.winpct_map.get(kA, 0.5), dtype=torch.float32)
        winB = torch.tensor(self.winpct_map.get(kB, 0.5), dtype=torch.float32)

        last10A = torch.tensor(self.last10_map.get(kA, 0.5), dtype=torch.float32)
        last10B = torch.tensor(self.last10_map.get(kB, 0.5), dtype=torch.float32)

        # confA = self.team_conf_map.get(A, None)
        # confB = self.team_conf_map.get(B, None)

        # confEffA = torch.tensor(
        #     self.conf_neteff_map.get((season, confA), DEFAULT_CONF_NETEFF),
        #     dtype=torch.float32
        # )
        # confEffB = torch.tensor(
        #     self.conf_neteff_map.get((season, confB), DEFAULT_CONF_NETEFF),
        #     dtype=torch.float32
        # )

        # confWP_A = torch.tensor(
        #     self.conf_winpct_map.get((season, confA), DEFAULT_CONF_WINPCT),
        #     dtype=torch.float32
        # )
        # confWP_B = torch.tensor(
        #     self.conf_winpct_map.get((season, confB), DEFAULT_CONF_WINPCT),
        #     dtype=torch.float32
        # )

        y = torch.tensor(float(r.y), dtype=torch.float32)

        return (
            seqA, seqB,
            seedA, seedB,
            winA, winB,
            last10A, last10B,
            # confEffA, confEffB,
            # confWP_A, confWP_B,
            y
        )


# 5) Model (GRU encoder + matchup head) + Brier training

In [39]:
class SeqEncoder(nn.Module):
    def __init__(self, feature_dim, hidden_dim=128, num_layers=2, dropout=0.1):
        super().__init__()
        self.rnn = nn.GRU(
            input_size=feature_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

    def forward(self, x):
        _, h = self.rnn(x)
        return h[-1]  # [B, H]

class MatchupModel(nn.Module):
    def __init__(self, feature_dim, hidden_dim=128, mlp_dim=128, dropout=0.25):
        super().__init__()
        self.enc = SeqEncoder(feature_dim, hidden_dim)

        # seed(4) + win(3) + last10(3) + conf(4) = 14 scalars
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 4 + 9, mlp_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, 1)
        )

    def forward(
        self, seqA, seqB,
        seedA, seedB,
        winA, winB,
        last10A, last10B,
        # confEffA, confEffB,
        # confWP_A, confWP_B
    ):
        embA = self.enc(seqA)
        embB = self.enc(seqB)

        seedA_s = 17.0 - seedA
        seedB_s = 17.0 - seedB

        extra = torch.stack([
            seedA_s, seedB_s, seedA_s - seedB_s,
            winA, winB, winA - winB,
            last10A, last10B, last10A - last10B,
            # confEffA, confEffB, confEffA - confEffB,
            # confWP_A, confWP_B
        ], dim=1)

        x = torch.cat([embA, embB, embA - embB, embA * embB, extra], dim=1)
        return self.head(x).squeeze(1)


def brier_loss_from_logits(logits, y):
    p = torch.sigmoid(logits)
    return torch.mean((p - y) ** 2)


### Train/valid loops

In [40]:
@torch.no_grad()
def eval_brier(model, loader, device):
    model.eval()
    total = 0.0
    n = 0

    for batch in loader:
        *inputs, y = batch
        inputs = [x.to(device) for x in inputs]
        y = y.to(device)

        logits = model(*inputs)
        loss = brier_loss_from_logits(logits, y)

        total += loss.item() * y.size(0)
        n += y.size(0)

    return total / n


import torch.nn.functional as torchF

def train_one_epoch(model, loader, optimizer, device, scheduler=None):
    model.train()
    total = 0.0
    n = 0

    for batch in loader:
        *inputs, y = batch
        inputs = [x.to(device, non_blocking=True) for x in inputs]
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(*inputs)
        loss = brier_loss_from_logits(logits, y)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        # OneCycleLR MUST step per batch, AFTER optimizer.step()
        if scheduler is not None:
            scheduler.step()

        total += loss.item() * y.size(0)
        n += y.size(0)

    return total / n


# 6) Train (single holdout season)

In [41]:
  print("train_pairs rows:", len(train_pairs))
  print("seasons in train_pairs:", train_pairs["Season"].min(), train_pairs["Season"].max())

  print("seednum_map size:", len(seednum_map))
  print("team_seq size:", len(team_seq))

  # sanity check a few keys
  sample = train_pairs.sample(5, random_state=1)
  for r in sample.itertuples():
      kA = (int(r.Season), int(r.TeamA))
      kB = (int(r.Season), int(r.TeamB))
      print(kA in seednum_map, kB in seednum_map, kA in team_seq, kB in team_seq, kA, kB)

train_pairs rows: 1788
seasons in train_pairs: 2010 2024
seednum_map size: 1744
team_seq size: 5602
True True True True (2021, 3329) (2021, 3448)
True True True True (2015, 3398) (2015, 3246)
True True True True (2016, 3417) (2016, 3218)
True True True True (2016, 3382) (2016, 3329)
True True True True (2021, 3376) (2021, 3210)


In [42]:
T = 12
F_len = len(feature_cols)

train_df = train_pairs[train_pairs["Season"].isin(train_seasons)].copy()
val_df   = train_pairs[train_pairs["Season"] == val_season].copy()

train_ds = MatchupSeqDataset(
    train_df, 
    team_seq, 
    seednum_map=seednum_map, 
    winpct_map=winpct_map,
    last10_map=last10_map,
    # team_conf_map=team_conf_map,
    # conf_neteff_map=conf_neteff_map,
    # conf_winpct_map=conf_winpct_map,
    T=T, 
    feature_dim=F_len
)
val_ds = MatchupSeqDataset(
    val_df, 
    team_seq, 
    seednum_map=seednum_map,
    winpct_map=winpct_map,
    last10_map=last10_map,
    # team_conf_map=team_conf_map,
    # conf_neteff_map=conf_neteff_map,
    # conf_winpct_map=conf_winpct_map,
    T=T, 
    feature_dim=F_len
)

# If you hit pickling issues in Kaggle notebooks, set num_workers=0
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=1024, shuffle=False, num_workers=0)


train_loader = DataLoader(train_ds, batch_size=512, shuffle=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=1024, shuffle=False, num_workers=0, pin_memory=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MatchupModel(feature_dim=F_len, hidden_dim=128, mlp_dim=128, dropout=0.25).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-4)
num_epochs = 51

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    opt,
    max_lr=3e-4,
    epochs=num_epochs,
    steps_per_epoch=len(train_loader),
    pct_start=0.3,
    anneal_strategy="cos",
    div_factor=25.0,        # start_lr = max_lr/div_factor = 4e-5
    final_div_factor=1e4
)


best_val = float("inf")
best_state = None

for epoch in range(1, num_epochs + 1):
    tr = train_one_epoch(model, train_loader, opt, device, scheduler=scheduler)
    va = eval_brier(model, val_loader, device)
    print(f"Epoch {epoch:02d} | train {tr:.5f} | val {va:.5f} | LR {opt.param_groups[0]['lr']:.2e}")


    if va < best_val:
        best_val = va
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

print("Best val Brier:", best_val)
if best_state is not None:
    model.load_state_dict(best_state)


Epoch 01 | train 0.25406 | val 0.25284 | LR 1.51e-05
Epoch 02 | train 0.25188 | val 0.25172 | LR 2.44e-05
Epoch 03 | train 0.25075 | val 0.24999 | LR 3.93e-05
Epoch 04 | train 0.25004 | val 0.24740 | LR 5.93e-05
Epoch 05 | train 0.24611 | val 0.24372 | LR 8.36e-05
Epoch 06 | train 0.24386 | val 0.23883 | LR 1.11e-04
Epoch 07 | train 0.23869 | val 0.23255 | LR 1.40e-04
Epoch 08 | train 0.23123 | val 0.22479 | LR 1.70e-04
Epoch 09 | train 0.22334 | val 0.21517 | LR 2.00e-04
Epoch 10 | train 0.21279 | val 0.20346 | LR 2.27e-04
Epoch 11 | train 0.20126 | val 0.18969 | LR 2.52e-04
Epoch 12 | train 0.18915 | val 0.17601 | LR 2.72e-04
Epoch 13 | train 0.17583 | val 0.16496 | LR 2.87e-04
Epoch 14 | train 0.16760 | val 0.15844 | LR 2.97e-04
Epoch 15 | train 0.16271 | val 0.15431 | LR 3.00e-04
Epoch 16 | train 0.16039 | val 0.14903 | LR 2.99e-04
Epoch 17 | train 0.15722 | val 0.14557 | LR 2.98e-04
Epoch 18 | train 0.15404 | val 0.14294 | LR 2.95e-04
Epoch 19 | train 0.15175 | val 0.13974 | LR 2.

In [43]:
train_pairs["Season"].max()

np.int64(2024)

In [44]:
# --- 2025 prediction setup (uses trained model) ---
pred_season = 2025
sub = pd.read_csv(sub_path)

def parse_id(row_id):
    season, teamA, teamB = row_id.split('_')
    return int(season), int(teamA), int(teamB)

tmp = sub['ID'].apply(parse_id)
sub['Season'] = tmp.apply(lambda x: x[0])
sub['TeamA'] = tmp.apply(lambda x: x[1])
sub['TeamB'] = tmp.apply(lambda x: x[2])

pred_pairs = sub[sub['Season'] == pred_season][['Season','TeamA','TeamB']].copy()

# Split men vs women by TeamID (<3000 men, >=3000 women)
pred_pairs_m = pred_pairs[(pred_pairs['TeamA'] < 3000) & (pred_pairs['TeamB'] < 3000)].copy()
pred_pairs_w = pred_pairs[(pred_pairs['TeamA'] >= 3000) & (pred_pairs['TeamB'] >= 3000)].copy()

# Use women's pairs for now
pred_pairs = pred_pairs_w

pred_pairs['DayNum'] = 135  # dummy; not used by the dataset
pred_pairs['y'] = 0.5      # dummy label required by MatchupSeqDataset

pred_ds = MatchupSeqDataset(
    pred_pairs,
    team_seq,
    seednum_map=seednum_map,
    winpct_map=winpct_map,
    last10_map=last10_map,
    T=T,
    feature_dim=F_len
)

pred_loader = DataLoader(pred_ds, batch_size=1024, shuffle=False, num_workers=0)
print('Prediction rows:', len(pred_ds), 'of', len(pred_pairs))


Prediction rows: 2278 of 65341


In [45]:
@torch.no_grad()
def predict_probs(model, loader, device):
    model.eval()
    all_probs = []
    for batch in loader:
        *inputs, _ = batch
        inputs = [x.to(device) for x in inputs]
        logits = model(*inputs)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.append(probs)
    return np.concatenate(all_probs)

device = next(model.parameters()).device
pred_probs = predict_probs(model, pred_loader, device)

pred_df = pred_ds.df.copy()
pred_df['ID'] = pred_df.apply(lambda r: f"{int(r.Season)}_{int(r.TeamA)}_{int(r.TeamB)}", axis=1)
pred_df['Pred'] = pred_probs

sub_out = sub[(sub['Season'] == pred_season) & (sub['TeamA'] >= 3000) & (sub['TeamB'] >= 3000)][['ID']].merge(
    pred_df[['ID','Pred']], on='ID', how='left'
)
missing = sub_out['Pred'].isna().sum()
if missing:
    print(f'WARNING: {missing} rows missing predictions (filled with 0.5).')
    sub_out['Pred'] = sub_out['Pred'].fillna(0.5)

sub_out.to_csv('submission_2025_womens.csv', index=False)
sub_out.head()


,ID,Pred
0,2025_3101_3102,0.5
1,2025_3101_3103,0.5
2,2025_3101_3104,0.5
3,2025_3101_3105,0.5
4,2025_3101_3106,0.5


In [46]:
pred_df

,Season,TeamA,TeamB,DayNum,y,ID,Pred
0,2025,3104,3117,135,0.5,2025_3104_3117,0.914048
1,2025,3104,3123,135,0.5,2025_3104_3123,0.868164
2,2025,3104,3124,135,0.5,2025_3104_3124,0.378297
3,2025,3104,3143,135,0.5,2025_3104_3143,0.783439
4,2025,3104,3162,135,0.5,2025_3104_3162,0.693087
...,...,...,...,...,...,...,...
2273,2025,3452,3456,135,0.5,2025_3452_3456,0.922762
2274,2025,3452,3471,135,0.5,2025_3452_3471,0.927023
2275,2025,3453,3456,135,0.5,2025_3453_3456,0.780211
2276,2025,3453,3471,135,0.5,2025_3453_3471,0.738352


In [47]:
sub

,ID,Pred,Season,TeamA,TeamB
0,2025_1101_1102,0.5,2025,1101,1102
1,2025_1101_1103,0.5,2025,1101,1103
2,2025_1101_1104,0.5,2025,1101,1104
3,2025_1101_1105,0.5,2025,1101,1105
4,2025_1101_1106,0.5,2025,1101,1106
...,...,...,...,...,...
131402,2025_3477_3479,0.5,2025,3477,3479
131403,2025_3477_3480,0.5,2025,3477,3480
131404,2025_3478_3479,0.5,2025,3478,3479
131405,2025_3478_3480,0.5,2025,3478,3480
